In [1]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
load_dotenv()
import os
import asyncio
api_key = os.getenv('OPENAI_API_KEY')
openai_model_client = OpenAIChatCompletionClient(
    model = "gpt-4o",
    api_key=api_key
)

In [2]:
from autogen_agentchat.agents import AssistantAgent

dsa_solver = AssistantAgent(
    name = 'Complex_DSA_Solver',
    model_client=openai_model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words"
)

code_reviewer = AssistantAgent(
    name = 'CODE_REVEIWER',
    model_client=openai_model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized.Give under 10 words.If the code is fine, please say 'TERMINATE'"
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=openai_model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required.Give under 10 words. If the code is fine, please say 'TERMINATE'"
)

In [7]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination

termination_condition = TextMentionTermination('TERMINATE')

teams = RoundRobinGroupChat(participants=[dsa_solver,code_reviewer,code_editor],termination_condition=termination_condition)

async def test_team():
    task = TextMessage(content = "Write a code for sort an array using buble sort.",source='User')

    result = await teams.run(task= task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")
        print("\n\n")


await test_team()



User : Write a code for sort an array using buble sort.



Complex_DSA_Solver : ```python
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

# Example usage
array = [64, 34, 25, 12, 22, 11, 90]
sorted_array = bubble_sort(array)
print("Sorted array:", sorted_array)
```
This code sorts an array using the bubble sort algorithm by repeatedly swapping adjacent elements if they are in the wrong order.



CODE_REVEIWER : TERMINATE



